In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-airbnb/newyork-city-airbnb-eda-small/checkpoints/pre_cell_14.pickle

In [4]:
%%RecordEvent
%%time
### cell 14 ###

# cell 14 optimized with single groupby and vectorized cudf operations
mask = (df.price > 50) & (df.price < 250)
df_tmp = df.assign(
    mask=mask,
    price_mask=df.price * mask
)
# one groupby to compute overall stats and in-range counts/sums
group_stats = df_tmp.groupby('neighbourhood_group').agg(
    count=("price", "count"),
    mean=("price", "mean"),
    count_50_250=("mask", "sum"),
    sum_price_50_250=("price_mask", "sum")
)
# compute mean for the 50–250 range
group_stats["mean_50_250"] = group_stats["sum_price_50_250"] / group_stats["count_50_250"]
# reorder to match nyc_sub_2
group_stats = group_stats.loc[nyc_sub_2]
# extract Python lists
total_amount_sub         = group_stats["count"].fillna(0).astype(int).tolist()
total_amount_sub_50_250  = group_stats["count_50_250"].fillna(0).astype(int).tolist()
aver_price_total        = group_stats["mean"].round(3).tolist()
aver_price_50_250       = group_stats["mean_50_250"].round(3).tolist()

total_amount_sub, total_amount_sub_50_250, aver_price_total, aver_price_50_250

CPU times: user 82.9 ms, sys: 19.8 ms, total: 103 ms
Wall time: 103 ms


([135590, 122600, 42980, 9140, 3310],
 [98450, 92400, 29720, 5730, 2250],
 [214.202, 132.852, 100.03, 89.008, 114.23],
 [137.576, 113.951, 99.322, 92.529, 99.213])

In [ ]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-airbnb/rewritten/q20_small_rewrite/checkpoints/post_cell_14_try_1.pickle